# Stage 1: Static ASL Alphabet Recognition
## CSE 474 - Introduction to Machine Learning, Spring 2026
### Team: Nitin Suresh Kumar, Yash Sabale, Vanshaj Arora

This notebook trains a MobileNetV2 model on the ASL Alphabet dataset for static gesture recognition.

**Dataset:** Kaggle ASL Alphabet (87,000 images, 29 classes)

**Expected Results:** ~99.66% validation accuracy

In [ ]:

import os, time, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


SEED              = 42
DATA_DIR          = Path("/kaggle/input/asl-alphabet/asl_alphabet_train/asl_alphabet_train")
OUTPUT_DIR        = Path("/kaggle/working")  # or Path("./outputs")
DEVICE            = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG_SIZE          = 224
BATCH_SIZE        = 32
EPOCHS            = 10
LR                = 1e-3
SAMPLES_PER_CLASS = 150  

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"Device: {DEVICE}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
classes = sorted([d.name for d in DATA_DIR.iterdir() if d.is_dir()])
print(f"Found {len(classes)} classes: {classes}")

all_paths, all_labels = [], []
for cls in classes:
    imgs = list((DATA_DIR / cls).glob("*.jpg"))
    if SAMPLES_PER_CLASS:
        imgs = random.sample(imgs, min(SAMPLES_PER_CLASS, len(imgs)))
    all_paths  += [str(p) for p in imgs]
    all_labels += [cls] * len(imgs)

le = LabelEncoder()
all_labels_enc = le.fit_transform(all_labels)
NUM_CLASSES = len(le.classes_)

print(f"Total samples: {len(all_paths)} | Classes: {NUM_CLASSES}")
print(f"Label mapping: {dict(zip(le.classes_, range(NUM_CLASSES)))}")

train_paths, val_paths, train_labels, val_labels = train_test_split(
    all_paths, all_labels_enc,
    test_size=0.2, random_state=SEED, stratify=all_labels_enc
)
print(f"Train: {len(train_paths)} | Val: {len(val_paths)}")

In [ ]:
class ASLDataset(Dataset):
    def __init__(self, paths, labels, transform):
        self.paths = paths
        self.labels = labels
        self.transform = transform
    
    def __len__(self):
        return len(self.paths)
    
    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        return self.transform(img), int(self.labels[idx])

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.RandomRotation(8),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_loader = DataLoader(
    ASLDataset(train_paths, train_labels, train_transform),
    batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True
)
val_loader = DataLoader(
    ASLDataset(val_paths, val_labels, val_transform),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True
)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")
print("Dataloaders ready!")

In [ ]:
model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)

model.classifier[1] = nn.Linear(model.last_channel, NUM_CLASSES)
model = model.to(DEVICE)

for param in model.parameters():
    param.requires_grad = False
for param in model.classifier.parameters():
    param.requires_grad = True

optimizer = optim.Adam(model.classifier.parameters(), lr=LR)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=4, gamma=0.5)
criterion = nn.CrossEntropyLoss()

print(f"Model: MobileNetV2")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print("Model ready!")

In [ ]:
def run_epoch(model, loader, optimizer=None, train=True):
    """Run one epoch of training or validation."""
    model.train() if train else model.eval()
    total_loss = correct = total = 0
    
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for X, y in loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            
            if train:
                optimizer.zero_grad()
            
            outputs = model(X)
            loss = criterion(outputs, y)
            
            if train:
                loss.backward()
                optimizer.step()
            
            total_loss += loss.item() * len(y)
            correct += (outputs.argmax(1) == y).sum().item()
            total += len(y)
    
    return total_loss / total, correct / total

history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

print("Starting training...")
print("="*70)

for epoch in range(1, EPOCHS + 1):
    if epoch == 4:
        print("\n>> Unfreezing full network for fine-tuning")
        for param in model.parameters():
            param.requires_grad = True
        optimizer = optim.Adam(model.parameters(), lr=LR / 10)
    
    t0 = time.time()
    
    train_loss, train_acc = run_epoch(model, train_loader, optimizer, train=True)
    
    val_loss, val_acc = run_epoch(model, val_loader, train=False)
    
    scheduler.step()
    
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc * 100)
    history["val_acc"].append(val_acc * 100)
    
    print(f"Epoch {epoch:2d}/{EPOCHS} | "
          f"Train Loss: {train_loss:.4f} Acc: {train_acc*100:.2f}% | "
          f"Val Loss: {val_loss:.4f} Acc: {val_acc*100:.2f}% | "
          f"Time: {time.time()-t0:.1f}s")

print("="*70)
print("Training complete!")

In [ ]:
model.eval()
all_preds, all_true = [], []

with torch.no_grad():
    for X, y in val_loader:
        preds = model(X.to(DEVICE)).argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_true.extend(y.numpy())

final_acc = accuracy_score(all_true, all_preds)
print(f"\nFinal Validation Accuracy: {final_acc*100:.2f}%")
print("\nClassification Report:")
print(classification_report(all_true, all_preds, target_names=le.classes_))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history["train_loss"], marker="o", label="Train Loss", linewidth=2)
ax1.plot(history["val_loss"], marker="s", label="Val Loss", linewidth=2)
ax1.set_xlabel("Epoch", fontsize=12)
ax1.set_ylabel("Loss", fontsize=12)
ax1.set_title("Training and Validation Loss", fontsize=14, fontweight="bold")
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)
ax1.set_xticks(range(len(history["train_loss"])))
ax1.set_xticklabels(range(1, len(history["train_loss"]) + 1))

ax2.plot(history["train_acc"], marker="o", label="Train Accuracy", linewidth=2)
ax2.plot(history["val_acc"], marker="s", label="Val Accuracy", linewidth=2)
ax2.set_xlabel("Epoch", fontsize=12)
ax2.set_ylabel("Accuracy (%)", fontsize=12)
ax2.set_title("Training and Validation Accuracy", fontsize=14, fontweight="bold")
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)
ax2.set_xticks(range(len(history["train_acc"])))
ax2.set_xticklabels(range(1, len(history["train_acc"]) + 1))

plt.suptitle("MobileNetV2 Training on ASL Alphabet (29 Classes)", 
             fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

cm = confusion_matrix(all_true, all_preds)
plt.figure(figsize=(16, 14))
sns.heatmap(cm, xticklabels=le.classes_, yticklabels=le.classes_,
            fmt="d", cmap="Blues", linewidths=0.5, annot=True, 
            annot_kws={"size": 8})
plt.title(f"Confusion Matrix | Validation Accuracy: {final_acc*100:.2f}%", 
          fontsize=16, fontweight="bold")
plt.xlabel("Predicted Label", fontsize=12)
plt.ylabel("True Label", fontsize=12)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nPlots saved to {OUTPUT_DIR}")

In [ ]:
model_save_path = OUTPUT_DIR / "asl_mobilenetv2.pth"
torch.save(model.state_dict(), model_save_path)
print(f"Model saved to: {model_save_path}")

import pickle
with open(OUTPUT_DIR / "label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)
print(f"Label encoder saved to: {OUTPUT_DIR / 'label_encoder.pkl'}")

print("\n" + "="*70)
print("STAGE 1 COMPLETE!")
print("="*70)
print(f"Final Accuracy: {final_acc*100:.2f}%")
print(f"Model saved to: {model_save_path}")
print("\nNext: Download the model and use it with the real-time demo:")
print("  python demo/realtime_asl_demo.py --model models/asl_mobilenetv2.pth")